<a href="https://colab.research.google.com/github/AlperYildirim1/geometric-grokking/blob/main/Fourier_Analysis_Last.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import json

# =============================================================================
# CONFIGURATION & SETUP
# =============================================================================
SAVE_DIR = '/content/drive/MyDrive/grokking_fourier_analysis_last'
os.makedirs(SAVE_DIR, exist_ok=True)

P = 113
FRAC_TRAIN = 0.3
D_MODEL = 128
NUM_HEADS = 4
MLP_DIM = 512
LR = 6e-4
MAX_EPOCHS = 100000
LOG_EVERY = 200
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def make_dataset(p, frac_train, seed=42):
    set_seed(seed)
    rng = random.Random(seed)
    all_pairs = [(a, b) for a in range(p) for b in range(p)]
    rng.shuffle(all_pairs)
    n_train = int(len(all_pairs) * frac_train)
    train_x = torch.tensor([[a, b, p] for a, b in all_pairs[:n_train]], dtype=torch.long)
    train_y = torch.tensor([(a + b) % p for a, b in all_pairs[:n_train]], dtype=torch.long)
    test_x = torch.tensor([[a, b, p] for a, b in all_pairs[n_train:]], dtype=torch.long)
    test_y = torch.tensor([(a + b) % p for a, b in all_pairs[n_train:]], dtype=torch.long)
    return train_x, train_y, test_x, test_y

# =============================================================================
# CORRECTED ARCHITECTURE (From Script 1)
# =============================================================================
class UnifiedTransformer(nn.Module):
    def __init__(self, p, d_model, num_heads, mlp_dim, model_type):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.model_type = model_type

        self.tok_embed = nn.Embedding(p + 1, d_model)
        self.pos_embed = nn.Embedding(3, d_model)

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        self.mlp_in = nn.Linear(d_model, mlp_dim)
        self.mlp_out = nn.Linear(mlp_dim, d_model)
        self.unembed = nn.Linear(d_model, p, bias=False)

        if self.model_type == 'layernorm':
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)
            self.norm3 = nn.LayerNorm(d_model)

    def apply_norm(self, x, norm_module=None):
        if self.model_type == 'layernorm':
            return norm_module(x)
        else:
            return F.normalize(x, dim=-1)

    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.tok_embed(x) + self.pos_embed(pos)

        h = self.apply_norm(h, getattr(self, 'norm1', None))

        Q = self.W_Q(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_K(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        V = self.W_V(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn_out = self.W_O((F.softmax(scores, dim=-1) @ V).transpose(1, 2).contiguous().view(B, L, self.d_model))

        h = h + attn_out
        h = self.apply_norm(h, getattr(self, 'norm2', None))

        h = h + self.mlp_out(F.relu(self.mlp_in(h)))
        h = self.apply_norm(h, getattr(self, 'norm3', None))

        h_final = h[:, 2, :]

        if self.model_type == 'spherical_nowd':
            w_normalized = F.normalize(self.unembed.weight, dim=1)
            logits = F.linear(h_final, w_normalized)
            return logits * 10.0
        else:
            return self.unembed(h_final)

# FOURIER DIAGNOSTICS
def extract_WL(model):
    if model.model_type == 'spherical_nowd':
        W_U_norm = F.normalize(model.unembed.weight, dim=1)
        return W_U_norm @ model.mlp_out.weight
    else:
        return model.unembed.weight @ model.mlp_out.weight

def plot_sines_and_cosines(model, p, title, seed):
    print(f"\n--- Generating SVD Chart for {title} ---")
    model.eval()
    with torch.no_grad():
        W_L = extract_WL(model)
        W_L_centered = W_L - W_L.mean(dim=0, keepdim=True)
        U, S, V = torch.svd(W_L_centered)

    fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
    vocab = torch.arange(p).cpu().numpy()

    for i in range(4):
        wave = U[:, i].cpu().numpy()
        axes[i].plot(vocab, wave, marker='o', markersize=3, linestyle='-', color='blue')
        axes[i].set_ylabel(f"SVD\nVector {i+1}")
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Vocabulary Token (0 to 112)")
    fig.suptitle(f"Top SVD Components of $W_L$: {title}", fontsize=14)
    plt.tight_layout()

    safe_name = title.replace(" ", "_").replace("(", "").replace(")", "").replace("=", "")
    save_path = os.path.join(SAVE_DIR, f"svd_{safe_name}_seed{seed}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Saved SVD plot to: {save_path}")
    plt.close(fig)

def run_fourier_diagnostics(model, name, p, test_x, test_y):
    print(f"\n{'='*60}")
    print(f" FOURIER DIAGNOSTICS: {name.upper()}")
    print(f"{'='*60}")
    model.eval()
    test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)

    W_L = extract_WL(model)
    fft_W_L = torch.fft.fft(W_L, dim=0)
    norms = torch.linalg.norm(torch.abs(fft_W_L), dim=1)[:p // 2 + 1]
    norms[0] = 0

    top_k = torch.argsort(norms, descending=True)[:5].cpu().numpy().tolist()
    print(f" Top 5 Key Frequencies detected in W_L: {top_k}")

    with torch.no_grad():
        orig_logits = model(test_x)
        fft_logits = torch.fft.rfft(orig_logits, dim=-1)

        mask = torch.zeros(fft_logits.shape[-1], dtype=torch.bool, device=DEVICE)
        mask[0] = True
        for k in top_k:
            if k < len(mask): mask[k] = True

        fft_restricted = fft_logits.clone()
        fft_restricted[:, ~mask] = 0
        logits_restricted = torch.fft.irfft(fft_restricted, n=p, dim=-1)
        restricted_acc = (logits_restricted.argmax(-1) == test_y).float().mean().item()
    print(f"🔪 Causal Ablation Acc (Only Top 5 Freqs): {restricted_acc*100:>6.2f}%")

    cache = {}
    def hook(m, i, o): cache['mlp'] = o.detach()
    handle = model.mlp_in.register_forward_hook(hook)

    a_vals, b_vals = torch.arange(p), torch.arange(p)
    A, B = torch.meshgrid(a_vals, b_vals, indexing='ij')
    inputs = torch.stack([A.flatten(), B.flatten(), torch.full_like(A.flatten(), p)], dim=1).to(DEVICE)

    with torch.no_grad():
        _ = model(inputs)
        mlp_acts = F.relu(cache['mlp'])[:, 2, :]
    handle.remove()

    c_vals = torch.arange(p).float().to(DEVICE)
    a_flat, b_flat = A.flatten().float(), B.flatten().float()

    print(f"\n{'Freq':<6} | {'2D FVE (Trig Identities)':<25}")
    print("-" * 35)

    for k in top_k:
        wk = 2 * math.pi * k / p
        cos_c, sin_c = torch.cos(wk * c_vals), torch.sin(wk * c_vals)
        u_k, v_k = cos_c @ W_L, sin_c @ W_L
        act_cos, act_sin = mlp_acts @ u_k, mlp_acts @ v_k

        ideal_cos = torch.cos(wk * (a_flat + b_flat)).to(DEVICE)
        ideal_sin = torch.sin(wk * (a_flat + b_flat)).to(DEVICE)

        def calc_fve_2d(actual, ideal_cos, ideal_sin):
            y = actual - actual.mean()
            x1 = ideal_cos - ideal_cos.mean()
            x2 = ideal_sin - ideal_sin.mean()
            X = torch.stack([x1, x2], dim=1)
            beta = torch.linalg.lstsq(X, y).solution
            fve = 1.0 - (torch.var(y - (X @ beta)) / torch.var(y))
            return max(0.0, fve.item() * 100.0)

        fve_u = calc_fve_2d(act_cos, ideal_cos, ideal_sin)
        fve_v = calc_fve_2d(act_sin, ideal_cos, ideal_sin)
        print(f"{k:<6} | U: {fve_u:>6.2f}%  V: {fve_v:>6.2f}%")


# TRAINING LOOP
def train_model(model, name, train_x, train_y, test_x, test_y, epochs, weight_decay):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=weight_decay, betas=(0.9, 0.999))
    criterion = nn.CrossEntropyLoss()

    train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
    test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)

    grok_epoch = None
    history = {'epochs': [], 'train_acc': [], 'test_acc': [], 'train_loss': [], 'test_loss': []}
    pbar = tqdm(range(epochs), desc=f"Training {name}")

    for epoch in pbar:
        model.train()
        logits = model(train_x)
        loss = criterion(logits, train_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % LOG_EVERY == 0 or epoch == epochs - 1:
            model.eval()
            with torch.no_grad():
                train_acc = (logits.argmax(-1) == train_y).float().mean().item()
                test_logits = model(test_x)
                test_loss = criterion(test_logits, test_y).item()
                test_acc = (test_logits.argmax(-1) == test_y).float().mean().item()

            history['epochs'].append(epoch)
            history['train_acc'].append(train_acc)
            history['test_acc'].append(test_acc)
            history['train_loss'].append(loss.item())
            history['test_loss'].append(test_loss)

            pbar.set_postfix({'tr_acc': f"{train_acc:.3f}", 'te_acc': f"{test_acc:.3f}"})

            # RESTORED 99% THRESHOLD
            if grok_epoch is None and test_acc >= 0.99:
                grok_epoch = epoch
                tqdm.write(f"\n {name} generalized (>99%) at epoch {epoch}!")
                break

    if grok_epoch is None:
        print(f"\n {name} failed to reach 99% generalization within {epochs} epochs.")

    return {"grok_epoch": grok_epoch if grok_epoch else f">{epochs}", "history": history}

# EXECUTION
if __name__ == "__main__":
    print("=" * 60)
    print("GROKKING REPRODUCTION: UNIFIED SCRIPT")
    print("=" * 60)

    for SEED in [42]:
        print(f"\n\n{'=' * 40}")
        print(f" STARTING RUN FOR SEED {SEED} ")
        print(f"{'=' * 40}\n")

        tr_x, tr_y, te_x, te_y = make_dataset(P, FRAC_TRAIN, seed=SEED)

        configs = [
            {"name": "LayerNorm Baseline", "type": "layernorm", "wd": 1.0},
            {"name": "Spherical Model (WD=1.0)", "type": "spherical_wd", "wd": 1.0},
            {"name": "Spherical Model (WD=0.0)", "type": "spherical_nowd", "wd": 0.0}
        ]

        results = {}
        for cfg in configs:
            set_seed(SEED)
            model = UnifiedTransformer(P, D_MODEL, NUM_HEADS, MLP_DIM, model_type=cfg["type"]).to(DEVICE)

            res = train_model(model, cfg["name"], tr_x, tr_y, te_x, te_y, MAX_EPOCHS, weight_decay=cfg["wd"])
            results[cfg["name"]] = res

            # Run Diagnostics
            run_fourier_diagnostics(model, cfg["name"], P, te_x, te_y)
            plot_sines_and_cosines(model, P, cfg["name"], SEED)

            # Save JSON
            safe_name = cfg["name"].replace(" ", "_").replace("(", "").replace(")", "").replace("=", "")
            save_path = os.path.join(SAVE_DIR, f"{safe_name}_seed{SEED}.json")
            with open(save_path, "w") as f:
                json.dump(res["history"], f)

        # Plot Accuracy History
        fig, axs = plt.subplots(1, 3, figsize=(18, 5))
        for i, (name, res) in enumerate(results.items()):
            hist = res["history"]
            axs[i].plot(hist["epochs"], hist["train_acc"], label="Train Acc", color="#1f77b4")
            axs[i].plot(hist["epochs"], hist["test_acc"], label="Test Acc", color="#d62728")
            axs[i].set_title(f"{name}\nGeneralized: {res['grok_epoch']}")
            axs[i].set_xlabel("Epochs")
            axs[i].set_ylabel("Accuracy")
            axs[i].grid(True, alpha=0.3)
            axs[i].legend()

        plt.tight_layout()
        plot_path = os.path.join(SAVE_DIR, f"accuracy_grid_seed{SEED}.png")
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved Accuracy Grid to: {plot_path}")

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import json

# CONFIGURATION & SETUP
SAVE_DIR = '/content/drive/MyDrive/grokking_fourier_analysis_last_1e-4'
os.makedirs(SAVE_DIR, exist_ok=True)

P = 113
FRAC_TRAIN = 0.3
D_MODEL = 128
NUM_HEADS = 4
MLP_DIM = 512
LR = 1e-4
MAX_EPOCHS = 100000
LOG_EVERY = 200
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def make_dataset(p, frac_train, seed=42):
    set_seed(seed)
    rng = random.Random(seed)
    all_pairs = [(a, b) for a in range(p) for b in range(p)]
    rng.shuffle(all_pairs)
    n_train = int(len(all_pairs) * frac_train)
    train_x = torch.tensor([[a, b, p] for a, b in all_pairs[:n_train]], dtype=torch.long)
    train_y = torch.tensor([(a + b) % p for a, b in all_pairs[:n_train]], dtype=torch.long)
    test_x = torch.tensor([[a, b, p] for a, b in all_pairs[n_train:]], dtype=torch.long)
    test_y = torch.tensor([(a + b) % p for a, b in all_pairs[n_train:]], dtype=torch.long)
    return train_x, train_y, test_x, test_y


# ARCHITECTURE
class UnifiedTransformer(nn.Module):
    def __init__(self, p, d_model, num_heads, mlp_dim, model_type):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.model_type = model_type

        self.tok_embed = nn.Embedding(p + 1, d_model)
        self.pos_embed = nn.Embedding(3, d_model)

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        self.mlp_in = nn.Linear(d_model, mlp_dim)
        self.mlp_out = nn.Linear(mlp_dim, d_model)
        self.unembed = nn.Linear(d_model, p, bias=False)

        if self.model_type == 'layernorm':
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)
            self.norm3 = nn.LayerNorm(d_model)

    def apply_norm(self, x, norm_module=None):
        if self.model_type == 'layernorm':
            return norm_module(x)
        else:
            return F.normalize(x, dim=-1)

    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.tok_embed(x) + self.pos_embed(pos)

        h = self.apply_norm(h, getattr(self, 'norm1', None))

        Q = self.W_Q(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_K(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        V = self.W_V(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn_out = self.W_O((F.softmax(scores, dim=-1) @ V).transpose(1, 2).contiguous().view(B, L, self.d_model))

        h = h + attn_out
        h = self.apply_norm(h, getattr(self, 'norm2', None))

        h = h + self.mlp_out(F.relu(self.mlp_in(h)))
        h = self.apply_norm(h, getattr(self, 'norm3', None))

        h_final = h[:, 2, :]

        if self.model_type == 'spherical_nowd':
            w_normalized = F.normalize(self.unembed.weight, dim=1)
            logits = F.linear(h_final, w_normalized)
            return logits * 10.0
        else:
            return self.unembed(h_final)

# FOURIER DIAGNOSTICS
def extract_WL(model):
    if model.model_type == 'spherical_nowd':
        W_U_norm = F.normalize(model.unembed.weight, dim=1)
        return W_U_norm @ model.mlp_out.weight
    else:
        return model.unembed.weight @ model.mlp_out.weight

def plot_sines_and_cosines(model, p, title, seed):
    print(f"\n--- Generating SVD Chart for {title} ---")
    model.eval()
    with torch.no_grad():
        W_L = extract_WL(model)
        W_L_centered = W_L - W_L.mean(dim=0, keepdim=True)
        U, S, V = torch.svd(W_L_centered)

    fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
    vocab = torch.arange(p).cpu().numpy()

    for i in range(4):
        wave = U[:, i].cpu().numpy()
        axes[i].plot(vocab, wave, marker='o', markersize=3, linestyle='-', color='blue')
        axes[i].set_ylabel(f"SVD\nVector {i+1}")
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Vocabulary Token (0 to 112)")
    fig.suptitle(f"Top SVD Components of $W_L$: {title}", fontsize=14)
    plt.tight_layout()

    safe_name = title.replace(" ", "_").replace("(", "").replace(")", "").replace("=", "")
    save_path = os.path.join(SAVE_DIR, f"svd_{safe_name}_seed{seed}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Saved SVD plot to: {save_path}")
    plt.close(fig)

def run_fourier_diagnostics(model, name, p, test_x, test_y):
    print(f"\n{'='*60}")
    print(f" FOURIER DIAGNOSTICS: {name.upper()}")
    print(f"{'='*60}")
    model.eval()
    test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)

    W_L = extract_WL(model)
    fft_W_L = torch.fft.fft(W_L, dim=0)
    norms = torch.linalg.norm(torch.abs(fft_W_L), dim=1)[:p // 2 + 1]
    norms[0] = 0

    top_k = torch.argsort(norms, descending=True)[:5].cpu().numpy().tolist()
    print(f" Top 5 Key Frequencies detected in W_L: {top_k}")

    with torch.no_grad():
        orig_logits = model(test_x)
        fft_logits = torch.fft.rfft(orig_logits, dim=-1)

        mask = torch.zeros(fft_logits.shape[-1], dtype=torch.bool, device=DEVICE)
        mask[0] = True
        for k in top_k:
            if k < len(mask): mask[k] = True

        fft_restricted = fft_logits.clone()
        fft_restricted[:, ~mask] = 0
        logits_restricted = torch.fft.irfft(fft_restricted, n=p, dim=-1)
        restricted_acc = (logits_restricted.argmax(-1) == test_y).float().mean().item()
    print(f"🔪 Causal Ablation Acc (Only Top 5 Freqs): {restricted_acc*100:>6.2f}%")

    cache = {}
    def hook(m, i, o): cache['mlp'] = o.detach()
    handle = model.mlp_in.register_forward_hook(hook)

    a_vals, b_vals = torch.arange(p), torch.arange(p)
    A, B = torch.meshgrid(a_vals, b_vals, indexing='ij')
    inputs = torch.stack([A.flatten(), B.flatten(), torch.full_like(A.flatten(), p)], dim=1).to(DEVICE)

    with torch.no_grad():
        _ = model(inputs)
        mlp_acts = F.relu(cache['mlp'])[:, 2, :]
    handle.remove()

    c_vals = torch.arange(p).float().to(DEVICE)
    a_flat, b_flat = A.flatten().float(), B.flatten().float()

    print(f"\n{'Freq':<6} | {'2D FVE (Trig Identities)':<25}")
    print("-" * 35)

    for k in top_k:
        wk = 2 * math.pi * k / p
        cos_c, sin_c = torch.cos(wk * c_vals), torch.sin(wk * c_vals)
        u_k, v_k = cos_c @ W_L, sin_c @ W_L
        act_cos, act_sin = mlp_acts @ u_k, mlp_acts @ v_k

        ideal_cos = torch.cos(wk * (a_flat + b_flat)).to(DEVICE)
        ideal_sin = torch.sin(wk * (a_flat + b_flat)).to(DEVICE)

        def calc_fve_2d(actual, ideal_cos, ideal_sin):
            y = actual - actual.mean()
            x1 = ideal_cos - ideal_cos.mean()
            x2 = ideal_sin - ideal_sin.mean()
            X = torch.stack([x1, x2], dim=1)
            beta = torch.linalg.lstsq(X, y).solution
            fve = 1.0 - (torch.var(y - (X @ beta)) / torch.var(y))
            return max(0.0, fve.item() * 100.0)

        fve_u = calc_fve_2d(act_cos, ideal_cos, ideal_sin)
        fve_v = calc_fve_2d(act_sin, ideal_cos, ideal_sin)
        print(f"{k:<6} | U: {fve_u:>6.2f}%  V: {fve_v:>6.2f}%")


# TRAINING LOOP
def train_model(model, name, train_x, train_y, test_x, test_y, epochs, weight_decay):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=weight_decay, betas=(0.9, 0.999))
    criterion = nn.CrossEntropyLoss()

    train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
    test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)

    grok_epoch = None
    history = {'epochs': [], 'train_acc': [], 'test_acc': [], 'train_loss': [], 'test_loss': []}
    pbar = tqdm(range(epochs), desc=f"Training {name}")

    for epoch in pbar:
        model.train()
        logits = model(train_x)
        loss = criterion(logits, train_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % LOG_EVERY == 0 or epoch == epochs - 1:
            model.eval()
            with torch.no_grad():
                train_acc = (logits.argmax(-1) == train_y).float().mean().item()
                test_logits = model(test_x)
                test_loss = criterion(test_logits, test_y).item()
                test_acc = (test_logits.argmax(-1) == test_y).float().mean().item()

            history['epochs'].append(epoch)
            history['train_acc'].append(train_acc)
            history['test_acc'].append(test_acc)
            history['train_loss'].append(loss.item())
            history['test_loss'].append(test_loss)

            pbar.set_postfix({'tr_acc': f"{train_acc:.3f}", 'te_acc': f"{test_acc:.3f}"})

            # RESTORED 99% THRESHOLD
            if grok_epoch is None and test_acc >= 0.99:
                grok_epoch = epoch
                tqdm.write(f"\n {name} generalized (>99%) at epoch {epoch}!")
                break

    if grok_epoch is None:
        print(f"\n {name} failed to reach 99% generalization within {epochs} epochs.")

    return {"grok_epoch": grok_epoch if grok_epoch else f">{epochs}", "history": history}

# EXECUTION
if __name__ == "__main__":
    print("=" * 60)
    print("GROKKING REPRODUCTION: UNIFIED SCRIPT")
    print("=" * 60)

    for SEED in [42]:
        print(f"\n\n{'=' * 40}")
        print(f" STARTING RUN FOR SEED {SEED} ")
        print(f"{'=' * 40}\n")

        tr_x, tr_y, te_x, te_y = make_dataset(P, FRAC_TRAIN, seed=SEED)

        configs = [
            {"name": "LayerNorm Baseline", "type": "layernorm", "wd": 1.0},
            {"name": "Spherical Model (WD=1.0)", "type": "spherical_wd", "wd": 1.0},
            {"name": "Spherical Model (WD=0.0)", "type": "spherical_nowd", "wd": 0.0}
        ]

        results = {}
        for cfg in configs:
            set_seed(SEED)
            model = UnifiedTransformer(P, D_MODEL, NUM_HEADS, MLP_DIM, model_type=cfg["type"]).to(DEVICE)

            res = train_model(model, cfg["name"], tr_x, tr_y, te_x, te_y, MAX_EPOCHS, weight_decay=cfg["wd"])
            results[cfg["name"]] = res

            # Run Diagnostics
            run_fourier_diagnostics(model, cfg["name"], P, te_x, te_y)
            plot_sines_and_cosines(model, P, cfg["name"], SEED)

            # Save JSON
            safe_name = cfg["name"].replace(" ", "_").replace("(", "").replace(")", "").replace("=", "")
            save_path = os.path.join(SAVE_DIR, f"{safe_name}_seed{SEED}.json")
            with open(save_path, "w") as f:
                json.dump(res["history"], f)

        # Plot Accuracy History
        fig, axs = plt.subplots(1, 3, figsize=(18, 5))
        for i, (name, res) in enumerate(results.items()):
            hist = res["history"]
            axs[i].plot(hist["epochs"], hist["train_acc"], label="Train Acc", color="#1f77b4")
            axs[i].plot(hist["epochs"], hist["test_acc"], label="Test Acc", color="#d62728")
            axs[i].set_title(f"{name}\nGeneralized: {res['grok_epoch']}")
            axs[i].set_xlabel("Epochs")
            axs[i].set_ylabel("Accuracy")
            axs[i].grid(True, alpha=0.3)
            axs[i].legend()

        plt.tight_layout()
        plot_path = os.path.join(SAVE_DIR, f"accuracy_grid_seed{SEED}.png")
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved Accuracy Grid to: {plot_path}")